In [1]:
from inference_training import Configuration, ImageDataset
from inference_training import initCudaEnvironment, createTransforms
from inference_training import drawImageAndFeatureMasks
from inference_training import exportOnnxModel, writeONNXMeta, loadONNX
from inference_training import trainModel, saveModel, loadModel
from inference_training import createModelInstance, testInference
import os

In [2]:
initCudaEnvironment(numCudaDevices=1,
                    visibleCudaDevices="0",
                    clearCudaDeviceCount=False)

# base model

In [3]:
# train on the GPU or on the CPU, if a GPU is not available
config = Configuration()
print("Device: " + str(config.device))

def create_model(trainDirectory, testDirectory):
    
    config.setDatasetPaths(trainPath=trainDirectory, testPath=testDirectory)
    config.setFilePrefix("")
    config.setModelName("25_epoch_combination_overlay")
    config.setInputSizes(inputWidth=250, inputHeight=250)
    config.setInputCellSize(cellSizeM=0.25, minCellSizeM=0.1, maxCellSizeM=0.5)
    config.setVersion(20250121)
    config.setModelInfo(channels=3, numClasses=2+1,  # (1 + background)
                        bboxOverlap=True, bboxPerImage=250, reuseModel=False)
    config.setEpochs(25)
    description = "Model description"
    config.setOnnxInfo(producer="Tygron", description=description)
    
    config.addLegendEntry("Background", 0, "#00000000")
    config.addLegendEntry("Label name 1", 1, "#00ffbf")
    config.addLegendEntry("Label name 2", 2, "#12d900")
    
    config.setOnnxMetaData(scoreThreshold=0.2,
                           maskThreshold=0.3,
                           strideFraction=0.5)
    
    config.setTensorInfo(tensorName='input_A:RGB_normalized', batchAmount=1)
    trainingDataset = ImageDataset(config, True, createTransforms(False))
    testDataset = ImageDataset(config, False, createTransforms(False))
    
    print("Train Image count: "+str(trainingDataset.__len__()))
    print("Test Image count: "+str(testDataset.__len__()))
    
    if not trainingDataset.validateFiles(False):
        print("Inconsistent training dataset ")
        trainingDataset.validateFiles(True)
    
    if not testDataset.validateFiles(False):
        print("Inconsistent test dataset ")
        testDataset.validateFiles(True)
    
    print("Pytorch model name " + config.getPytorchModelFileName())
    print("Onnx file name " + config.getOnnxFileName())
    
    imageNumber = 5
    print(trainingDataset.getLabelList(imageNumber))
    drawImageAndFeatureMasks(config, trainingDataset, imageNumber)
    
    model = trainModel(config, trainingDataset, testDataset)
    saveModel(config, model, path="models/"+config.getPytorchModelFileName())
    
    model.eval()
    testPrediction = testInference(config, model=model,
                               dataset=testDataset, imageNumber=88)
    
    exportOnnxModel(config, model)
    writeONNXMeta(config)
    onnx_model = loadONNX(config)
    print(f"metadata_props={onnx_model.metadata_props}")

Device: cpu


C:\Users\Gebruiker\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\cuda\__init__.py:174: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11050). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10\cuda\CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


In [5]:
base_directory = "datasets/train_data"
trainDirectory = "datasets/train_data"
testDirectory = "datasets/test_data"

#create_model(trainDirectory, testDirectory)

In [6]:
#loadExistingModel = False

#if loadExistingModel:
#    model = createModelInstance(config)
#    loadModel(config, model, path=config.getPytorchModelFileName())

#else:
#    model = trainModel(config, trainingDataset, testDataset)
#    saveModel(config, model, path=config.getPytorchModelFileName())

In [8]:
model = createModelInstance(config)
config.setDatasetPaths(trainPath=trainDirectory, testPath=testDirectory)
testDataset = ImageDataset(config, False, createTransforms(False))
loadModel(config, model, path=config.getPytorchModelFileName())
model.eval()
testPrediction = testInference(config, model=model,
                               dataset=testDataset, imageNumber=20)

Detections per image 250
in features mask: 256


FileNotFoundError: [Errno 2] No such file or directory: 'models/inference_1c_25cm_20e_250_maxgt.pt'

In [ ]:
#exportOnnxModel(config, model)
#writeONNXMeta(config)
#onnx_model = loadONNX(config)
#print(f"metadata_props={onnx_model.metadata_props}")